### 라이브러리 임포트 및 데이터 불러오기

In [7]:
! pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 9.1 MB/s eta 0:00:00


In [8]:
### 필요한 패키지
import torch
import pickle
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import platform
import seaborn as sns
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, precision_recall_curve, confusion_matrix

In [9]:
# pkl 파일 열기
with open('/content/drive/MyDrive/Colab Notebooks/리스크 공모전/Health_Insurance_final.pkl', 'rb') as f:
    data = pickle.load(f)
print(data.head(5))

   lapse  exposure_time  premium  seniority_insured  seniority_policy  \
0      1       0.969749  2923.32                 24                24   
1      2       1.000000  1930.11                 24                24   
2      2       1.000000  2020.00                 24                24   
3      2       1.000000  1843.25                 24                24   
4      2       1.000000  1316.03                 24                24   

  type_policy_dg type_product new_business  cost_claims_year  \
0              I            S           No           8242.62   
1              I            S           No            344.41   
2              I            P           No            523.18   
3              I            S           No            257.55   
4              I            S           No            420.16   

   n_medical_services  ... IICIMUN_capped help_requested  high_loss  \
0                  32  ...      22.842273              1   2.819609   
1                  22  ...      22

In [10]:
data.columns.tolist()

['lapse',
 'exposure_time',
 'premium',
 'seniority_insured',
 'seniority_policy',
 'type_policy_dg',
 'type_product',
 'new_business',
 'cost_claims_year',
 'n_medical_services',
 'reimbursement',
 'distribution_channel',
 'age',
 'gender',
 'IICIMUN',
 'IICIPROV',
 'C_C',
 'C_H',
 'C_GI',
 'C_II',
 'C_IE_P',
 'C_IE_T',
 'C_GE_P',
 'C_GE_T',
 'ID',
 'period',
 'missing_geo_cxt',
 'is_churn',
 'cost_claims_year_capped',
 'n_medical_services_capped',
 'log_n_medical_services',
 'log_cost_claims_year',
 'C_H_num',
 'IICIMUN_capped',
 'help_requested',
 'high_loss',
 'relative_poverty',
 'kr_age_risk',
 'kr_premium_shock',
 'kr_shock_amount',
 'kr_economic_stress',
 'kr_early_laps',
 'kr_medical_desert']

In [11]:
# 결측 확인
data.isna().sum()

,0
lapse,0
exposure_time,0
premium,0
seniority_insured,0
seniority_policy,0
type_policy_dg,0
type_product,0
new_business,0
cost_claims_year,0
n_medical_services,0


In [12]:
# target 분포
data.is_churn.value_counts(normalize=True)

,proportion
is_churn,
0,0.819492
1,0.180508


- 데이터 분할

In [13]:
target = 'is_churn'

x_cols = [
    # 기본
    'premium', 'seniority_policy', 'type_policy_dg', 'type_product', 'new_business',
    'log_cost_claims_year', 'distribution_channel',
    # 나이 관련
    'age',
    # 성별 관련
    'gender',
    # 지역 관련
    'IICIMUN_capped', 'IICIPROV', 'C_C', 'C_H_num', 'C_GI', 'C_IE_T',
]

der_cols = [
    'missing_geo_cxt',       # 지역 결측 신호
    'high_loss',                  # 보험사 손해율
    'relative_poverty',        # 지역 내 상대적 빈곤
    'kr_premium_shock',   # 가격 인상 압박(비율 단위)
    'kr_economic_stress',  # 소득 대비 체감 부담
    # 'kr_retention_years',    # 장기 유지 혜택
    'kr_early_laps',             #신계약 위험 구간
    # 'kr_direct_channel',     # 가입 채널 영향
    'kr_medical_desert'     # 인프라 취약성
]

print(f"사용 변수 개수(기본) : {len(x_cols)}")
print(f"사용 변수 개수(파생) : {len(der_cols)}")
x_cols += der_cols
print(f"최종 변수 개수 : {len(x_cols)}개")

device = "cuda" if torch.cuda.is_available() else "cpu"

사용 변수 개수(기본) : 15
사용 변수 개수(파생) : 7
최종 변수 개수 : 22개


In [14]:
# Encoding
cat_cols = ['type_policy_dg', 'type_product', 'new_business', 'distribution_channel', 'gender', 'C_C', 'kr_early_laps']
for col in cat_cols:
    data[col] = data[col].astype('category')

In [15]:
# train/test split
train = data[data.period == 2017]
val = data[data.period == 2018]
test = data[data.period == 2019]

X_train, y_train = train[x_cols], train[target]
X_val, y_val = val[x_cols], val[target]
X_test, y_test = test[x_cols], test[target]

### threshold 튜닝 전

#### pr-auc 기준 하이퍼파라미터 튜닝

In [ ]:
# 1차 (500번 수행) : 범위가 너무 넓어 탐색 시간이 오래 걸리고, 성능 향상이 나타나지 않아 중간에 멈추고 범위 축소
def objective(trial):

    max_depth = trial.suggest_int('max_depth', 3, 15)
    max_leaves = min(2**max_depth, 1024)

    params = {
        'n_estimators': 2000,  # Early stopping을 믿고 넉넉하게 설정
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 2, max_leaves),
        'max_depth': max_depth,
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0), # 불균형 제어
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    # Early stopping 적용 (성능 향상이 없으면 50라운드 후 학습 조기 종료)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='pr_auc', # PR-AUC 평가 지표
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(0) # 로그 출력 숨김
        ]
    )

    preds_proba = model.predict_proba(X_val)[:, 1]
    prauc = average_precision_score(y_val, preds_proba)

    return prauc

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=1500) # 트리 모델에 맞춰 500회 탐색

print("\n" + "="*40)
print(f"최적 Val PR-AUC : {study.best_value:.4f}")
print(f"최적 파라미터 : {study.best_params}")
print("="*40)

[I 2026-05-10 09:13:20,866] A new study created in memory with name: no-name-3b0d7255-94d9-4678-bf0f-4cb300fef95c
[I 2026-05-10 09:13:21,278] Trial 0 finished with value: 0.34443897615275176 and parameters: {'max_depth': 7, 'learning_rate': 0.1540359659501924, 'num_leaves': 94, 'min_child_samples': 122, 'subsample': 0.4936111842654619, 'colsample_bytree': 0.49359671220172163, 'scale_pos_weight': 1.5227525095137953}. Best is trial 0 with value: 0.34443897615275176.
[I 2026-05-10 09:13:23,113] Trial 1 finished with value: 0.31563088964612646 and parameters: {'max_depth': 14, 'learning_rate': 0.02416482602989751, 'num_leaves': 726, 'min_child_samples': 9, 'subsample': 0.9819459112971965, 'colsample_bytree': 0.899465584480253, 'scale_pos_weight': 2.9110519961044856}. Best is trial 0 with value: 0.34443897615275176.
[I 2026-05-10 09:13:23,492] Trial 2 finished with value: 0.312782261085373 and parameters: {'max_depth': 5, 'learning_rate': 0.002642526057549917, 'num_leaves': 11, 'min_child_s

KeyboardInterrupt: 

In [ ]:
# 1차 탐색 기준 최고 파라미터 조합
print("현재까지 최고 Val PR-AUC:", study.best_value)
print("최고 파라미터 조합:\n", study.best_params)

현재까지 최고 Val PR-AUC: 0.34443897615275176
최고 파라미터 조합:
 {'max_depth': 7, 'learning_rate': 0.1540359659501924, 'num_leaves': 94, 'min_child_samples': 122, 'subsample': 0.4936111842654619, 'colsample_bytree': 0.49359671220172163, 'scale_pos_weight': 1.5227525095137953}


In [ ]:
# 2차 탐색
def objective(trial):
    max_depth = trial.suggest_int('max_depth', 5, 9)   # 최적값 7 중심으로 좁은 범위 설정
    max_leaves = min(2 ** max_depth, 150) # max_depth에 맞춘 유연한 잎의 개수
    min_leaves = min(70, max_leaves)
    params = {
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.08, 0.25), # 0.154 주변 정밀 탐색
        'max_depth': max_depth,
        'num_leaves': trial.suggest_int('num_leaves', min_leaves, max_leaves),
        'min_child_samples': trial.suggest_int('min_child_samples', 100, 160), # 122 주변 탐색
        'subsample': trial.suggest_float('subsample', 0.35, 0.65),         # 0.49 주변 탐색 (데이터와 피처를 절반 정도 사용)
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.35, 0.65),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 2.5), # 1.52 주변 탐색
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='pr_auc',
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(0)
        ]
    )

    preds_proba = model.predict_proba(X_val)[:, 1]
    prauc = average_precision_score(y_val, preds_proba)

    return prauc

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=500)

print("\n" + "="*40)
print(f"최종 정밀 탐색 최고 PR-AUC : {study.best_value:.4f}")
print(f"최적 파라미터 : {study.best_params}")
print("="*40)

[I 2026-05-10 09:33:27,428] A new study created in memory with name: no-name-aea674ff-2d1c-497d-9203-090978d3d178
[I 2026-05-10 09:33:29,281] Trial 0 finished with value: 0.3206114862894348 and parameters: {'max_depth': 6, 'learning_rate': 0.24162143208968573, 'num_leaves': 64, 'min_child_samples': 144, 'subsample': 0.529597545259111, 'colsample_bytree': 0.3968055921327309, 'scale_pos_weight': 1.233991780504304}. Best is trial 0 with value: 0.3206114862894348.
[I 2026-05-10 09:33:29,570] Trial 1 finished with value: 0.32236341347749425 and parameters: {'max_depth': 5, 'learning_rate': 0.227249944781739, 'num_leaves': 32, 'min_child_samples': 136, 'subsample': 0.5624217733388137, 'colsample_bytree': 0.3561753482887407, 'scale_pos_weight': 2.4548647782429915}. Best is trial 1 with value: 0.32236341347749425.
[I 2026-05-10 09:33:29,998] Trial 2 finished with value: 0.3267612167974169 and parameters: {'max_depth': 9, 'learning_rate': 0.11609764881530694, 'num_leaves': 84, 'min_child_sample


최종 정밀 탐색 최고 PR-AUC : 0.3781
최적 파라미터 : {'max_depth': 5, 'learning_rate': 0.2498855762212911, 'num_leaves': 32, 'min_child_samples': 140, 'subsample': 0.5299175899599644, 'colsample_bytree': 0.5593990379735718, 'scale_pos_weight': 1.183426483440904}


In [ ]:
# 3차 탐색
def objective(trial):
    max_depth = trial.suggest_int('max_depth', 3, 6)   # 트리를 더 얕게 만들 수 있도록 하한선을 3으로 낮춤
    max_leaves_limit = 2 ** max_depth    # max_depth에 따라 생성 가능한 최대 잎의 수

    params = {
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.20, 0.35),
        'max_depth': max_depth,
        'num_leaves': trial.suggest_int('num_leaves', 8, max_leaves_limit),     # 잎의 개수는 8개부터 최대 허용치까지 유연하게 탐색
        'min_child_samples': trial.suggest_int('min_child_samples', 120, 160),
        'subsample': trial.suggest_float('subsample', 0.45, 0.60),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.50, 0.65),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 1.4),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='pr_auc',
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(0)
        ]
    )

    preds_proba = model.predict_proba(X_val)[:, 1]
    prauc = average_precision_score(y_val, preds_proba)

    return prauc

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=1000)

print("\n" + "="*40)
print(f"최고 PR-AUC : {study.best_value:.4f}")
print(f"최적 파라미터 : {study.best_params}")
print("="*40)

[I 2026-05-10 09:39:19,163] A new study created in memory with name: no-name-aa15ca62-17c6-4cc1-9b6c-d9bd623f0d12
[I 2026-05-10 09:39:19,481] Trial 0 finished with value: 0.31542607272749573 and parameters: {'max_depth': 4, 'learning_rate': 0.3426071459614874, 'num_leaves': 14, 'min_child_samples': 144, 'subsample': 0.4734027960663655, 'colsample_bytree': 0.5233991780504303, 'scale_pos_weight': 1.0232334448672797}. Best is trial 0 with value: 0.31542607272749573.
[I 2026-05-10 09:39:19,789] Trial 1 finished with value: 0.31992315301486024 and parameters: {'max_depth': 6, 'learning_rate': 0.2901672517614813, 'num_leaves': 48, 'min_child_samples': 120, 'subsample': 0.5954864778242991, 'colsample_bytree': 0.6248663961200633, 'scale_pos_weight': 1.0849356442713105}. Best is trial 1 with value: 0.31992315301486024.
[I 2026-05-10 09:39:20,033] Trial 2 finished with value: 0.3284361471457098 and parameters: {'max_depth': 3, 'learning_rate': 0.22751067647801507, 'num_leaves': 8, 'min_child_sam


최고 PR-AUC : 0.4106
최적 파라미터 : {'max_depth': 5, 'learning_rate': 0.3471822878189745, 'num_leaves': 23, 'min_child_samples': 144, 'subsample': 0.5111636910456923, 'colsample_bytree': 0.5646394676662235, 'scale_pos_weight': 1.0061636591457408}


In [ ]:
# 4차 탐색
def objective(trial):
    max_depth = trial.suggest_int('max_depth', 4, 6)
    max_leaves_limit = 2 ** max_depth

    params = {
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.25, 0.45),
        'max_depth': max_depth,
        'num_leaves': trial.suggest_int('num_leaves', 15, max_leaves_limit),
        'min_child_samples': trial.suggest_int('min_child_samples', 130, 155),
        'subsample': trial.suggest_float('subsample', 0.45, 0.55),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.50, 0.60),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.9, 1.1),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='pr_auc',
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(0)
        ]
    )

    preds_proba = model.predict_proba(X_val)[:, 1]
    prauc = average_precision_score(y_val, preds_proba)

    return prauc

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=1500)

print("\n" + "="*40)
print(f"최고 PR-AUC : {study.best_value:.4f}")
print(f"최적 파라미터 : {study.best_params}")
print("="*40)

[I 2026-05-10 09:48:32,740] A new study created in memory with name: no-name-760ff073-653d-467d-bad9-81cf5cad4735
[I 2026-05-10 09:48:33,014] Trial 0 finished with value: 0.3477369601587254 and parameters: {'max_depth': 5, 'learning_rate': 0.4401428612819832, 'num_leaves': 28, 'min_child_samples': 145, 'subsample': 0.46560186404424364, 'colsample_bytree': 0.5155994520336202, 'scale_pos_weight': 0.9116167224336399}. Best is trial 0 with value: 0.3477369601587254.
[I 2026-05-10 09:48:33,325] Trial 1 finished with value: 0.33032821156207 and parameters: {'max_depth': 6, 'learning_rate': 0.37022300234864175, 'num_leaves': 50, 'min_child_samples': 130, 'subsample': 0.5469909852161995, 'colsample_bytree': 0.5832442640800422, 'scale_pos_weight': 0.9424678221356553}. Best is trial 0 with value: 0.3477369601587254.
[I 2026-05-10 09:48:33,582] Trial 2 finished with value: 0.34564272927703743 and parameters: {'max_depth': 4, 'learning_rate': 0.28668090197068674, 'num_leaves': 15, 'min_child_sampl


최고 PR-AUC : 0.3951
최적 파라미터 : {'max_depth': 5, 'learning_rate': 0.34356902610511464, 'num_leaves': 27, 'min_child_samples': 135, 'subsample': 0.5382177568191784, 'colsample_bytree': 0.5466675849154781, 'scale_pos_weight': 0.9767207219327285}


- 최고 PR-AUC : 0.4106
- 최적 파라미터 : {'max_depth': 5, 'learning_rate': 0.3471822878189745, 'num_leaves': 23, 'min_child_samples': 144, 'subsample': 0.5111636910456923, 'colsample_bytree': 0.5646394676662235, 'scale_pos_weight': 1.0061636591457408}

#### auc 기준 하이퍼파라미터 튜닝

In [ ]:
def objective_auc(trial):

    max_depth = trial.suggest_int('max_depth', 4, 10)
    max_leaves_limit = min(2 ** max_depth, 256)

    params = {
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': max_depth,
        'num_leaves': trial.suggest_int('num_leaves', 15, max_leaves_limit),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 160),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',    # eval_metric 'auc'로 변경
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(0)
        ]
    )

    preds_proba = model.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, preds_proba)    # PR-AUC 대신 ROC-AUC 계산

    return auc_score

print("ROC-AUC 기준 탐색을 시작합니다...")
study_auc = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))

study_auc.optimize(objective_auc, n_trials=1000)

print("\n" + "="*40)
print(f"최고 ROC-AUC : {study_auc.best_value:.4f}")
print(f"최적 파라미터 : {study_auc.best_params}")
print("="*40)

[I 2026-05-10 10:01:27,805] A new study created in memory with name: no-name-9267bef4-16a3-452b-ab31-476989481fd8


ROC-AUC 기준 탐색을 시작합니다...


[I 2026-05-10 10:01:28,232] Trial 0 finished with value: 0.7283995353808214 and parameters: {'max_depth': 6, 'learning_rate': 0.2536999076681772, 'num_leaves': 51, 'min_child_samples': 104, 'subsample': 0.4936111842654619, 'colsample_bytree': 0.49359671220172163, 'scale_pos_weight': 1.2323344486727978}. Best is trial 0 with value: 0.7283995353808214.
[I 2026-05-10 10:01:28,966] Trial 1 finished with value: 0.6631928299885524 and parameters: {'max_depth': 10, 'learning_rate': 0.07725378389307355, 'num_leaves': 186, 'min_child_samples': 22, 'subsample': 0.9819459112971965, 'colsample_bytree': 0.899465584480253, 'scale_pos_weight': 1.8493564427131046}. Best is trial 0 with value: 0.7283995353808214.
[I 2026-05-10 10:01:32,335] Trial 2 finished with value: 0.6563530894769439 and parameters: {'max_depth': 5, 'learning_rate': 0.018659959624904916, 'num_leaves': 20, 'min_child_samples': 93, 'subsample': 0.6591670111852694, 'colsample_bytree': 0.5747374841188252, 'scale_pos_weight': 3.44741157


최고 ROC-AUC : 0.7496
최적 파라미터 : {'max_depth': 5, 'learning_rate': 0.16846100748961326, 'num_leaves': 20, 'min_child_samples': 62, 'subsample': 0.657706798505073, 'colsample_bytree': 0.4886885979244237, 'scale_pos_weight': 1.1370046342002038}


- 최고 ROC-AUC : 0.7496
- 최적 파라미터 : {'max_depth': 5, 'learning_rate': 0.16846100748961326, 'num_leaves': 20, 'min_child_samples': 62, 'subsample': 0.657706798505073, 'colsample_bytree': 0.4886885979244237, 'scale_pos_weight': 1.1370046342002038}

#### f1_Score 기준 하이퍼파라미터 튜닝

In [ ]:
def objective_f1(trial):
    max_depth = trial.suggest_int('max_depth', 4, 10)
    max_leaves_limit = min(2 ** max_depth, 256)

    params = {
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': max_depth,
        'num_leaves': trial.suggest_int('num_leaves', 15, max_leaves_limit),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 160),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='binary_logloss',   # 기본값
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(0)
        ]
    )

    preds_proba = model.predict_proba(X_val)[:, 1]
    preds_class = (preds_proba > 0.5).astype(int)    # 기본 threshold = 0.5 이용해 f1_score 계산

    f1 = f1_score(y_val, preds_class)

    return f1

study_f1 = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_f1.optimize(objective_f1, n_trials=1000)

print("\n" + "="*40)
print(f"최고 F1-score : {study_f1.best_value:.4f}")
print(f"최적 파라미터 : {study_f1.best_params}")
print("="*40)

[I 2026-05-10 10:18:41,925] A new study created in memory with name: no-name-4b913ab9-21fe-4721-a1fe-fc3ae210527e
[I 2026-05-10 10:18:43,204] Trial 0 finished with value: 0.12631318749227538 and parameters: {'max_depth': 6, 'learning_rate': 0.2536999076681772, 'num_leaves': 51, 'min_child_samples': 104, 'subsample': 0.4936111842654619, 'colsample_bytree': 0.49359671220172163, 'scale_pos_weight': 1.2323344486727978}. Best is trial 0 with value: 0.12631318749227538.
[I 2026-05-10 10:18:43,863] Trial 1 finished with value: 0.0 and parameters: {'max_depth': 10, 'learning_rate': 0.07725378389307355, 'num_leaves': 186, 'min_child_samples': 22, 'subsample': 0.9819459112971965, 'colsample_bytree': 0.899465584480253, 'scale_pos_weight': 1.8493564427131046}. Best is trial 0 with value: 0.12631318749227538.
[I 2026-05-10 10:18:44,165] Trial 2 finished with value: 0.0 and parameters: {'max_depth': 5, 'learning_rate': 0.018659959624904916, 'num_leaves': 20, 'min_child_samples': 93, 'subsample': 0.6


최고 F1-score : 0.2697
최적 파라미터 : {'max_depth': 5, 'learning_rate': 0.29892386276823396, 'num_leaves': 24, 'min_child_samples': 97, 'subsample': 0.433787601904423, 'colsample_bytree': 0.7983142308040867, 'scale_pos_weight': 1.4238253021337417}


- 최고 F1-score : 0.2697
- 최적 파라미터 : {'max_depth': 5, 'learning_rate': 0.29892386276823396, 'num_leaves': 24, 'min_child_samples': 97, 'subsample': 0.433787601904423, 'colsample_bytree': 0.7983142308040867, 'scale_pos_weight': 1.4238253021337417}

-> f1-score 기준, 성능이 0.3에 미치지 못하는 poor한 모습이므로 threshold 조정해야 함!

#### threshold 조정 전 튜닝 결과 비교

In [ ]:
def evaluate_baseline_model(params, model_name="Model"):
    print(f"\n[{model_name} 기본(Threshold 0.5) 성능 평가]")

    # 1. 모델 학습
    model = LGBMClassifier(**params, n_estimators=2000, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
    )

    # 2. 확률값 예측
    preds_proba = model.predict_proba(X_val)[:, 1]

    # 기본 threshold
    default_threshold = 0.5
    final_preds = (preds_proba >= default_threshold).astype(int)

    # 성능 지표 계산
    roc_auc = roc_auc_score(y_val, preds_proba)
    pr_auc = average_precision_score(y_val, preds_proba)

    f1 = f1_score(y_val, final_preds)
    precision = precision_score(y_val, final_preds)
    recall = recall_score(y_val, final_preds)
    cm = confusion_matrix(y_val, final_preds)

    print("=" * 50)
    print(f"📊 {model_name} 성능 리포트 (Threshold: 0.5 고정)")
    print("=" * 50)
    print(f"• PR-AUC   : {pr_auc:.4f} (임계값 무관)")
    print(f"• ROC-AUC  : {roc_auc:.4f} (임계값 무관)")
    print("-" * 50)
    print(f"• F1-Score : {f1:.4f}")
    print(f"• Precision: {precision:.4f}")
    print(f"• Recall   : {recall:.4f}")
    print("-" * 50)
    print("📌 Confusion Matrix:")
    print(f"TN: {cm[0][0]} | FP: {cm[0][1]}")
    print(f"FN: {cm[1][0]} | TP: {cm[1][1]}")
    print("=" * 50)

params_prauc = {'max_depth': 5, 'learning_rate': 0.3471822878189745, 'num_leaves': 23, 'min_child_samples': 144, 'subsample': 0.5111636910456923, 'colsample_bytree': 0.5646394676662235, 'scale_pos_weight': 1.0061636591457408}
params_auc = {'max_depth': 5, 'learning_rate': 0.16846100748961326, 'num_leaves': 20, 'min_child_samples': 62, 'subsample': 0.657706798505073, 'colsample_bytree': 0.4886885979244237, 'scale_pos_weight': 1.1370046342002038}

evaluate_baseline_model(params_prauc, "PR-AUC 타겟 모델")
evaluate_baseline_model(params_auc, "ROC-AUC 타겟 모델")


[PR-AUC 타겟 모델 기본(Threshold 0.5) 성능 평가]
📊 PR-AUC 타겟 모델 성능 리포트 (Threshold: 0.5 고정)
• PR-AUC   : 0.4106 (임계값 무관)
• ROC-AUC  : 0.7404 (임계값 무관)
--------------------------------------------------
• F1-Score : 0.0099
• Precision: 0.6034
• Recall   : 0.0050
--------------------------------------------------
📌 Confusion Matrix:
TN: 59436 | FP: 46
FN: 13922 | TP: 70

[ROC-AUC 타겟 모델 기본(Threshold 0.5) 성능 평가]
📊 ROC-AUC 타겟 모델 성능 리포트 (Threshold: 0.5 고정)
• PR-AUC   : 0.3508 (임계값 무관)
• ROC-AUC  : 0.7081 (임계값 무관)
--------------------------------------------------
• F1-Score : 0.0507
• Precision: 0.5096
• Recall   : 0.0267
--------------------------------------------------
📌 Confusion Matrix:
TN: 59123 | FP: 359
FN: 13619 | TP: 373


### threshold 튜닝 후 성능지표

In [16]:
# PR-AUC 기준 타겟 파라미터
best_params = {
    'max_depth': 5,
    'learning_rate': 0.3471822878189745,
    'num_leaves': 23,
    'min_child_samples': 144,
    'subsample': 0.5111636910456923,
    'colsample_bytree': 0.5646394676662235,
    'scale_pos_weight': 1.0061636591457408,
    'n_estimators': 2000,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

# 모델 학습
model = LGBMClassifier(**best_params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
)

# ---- Validation: F1 최대 Threshold 탐색 ----
preds_proba = model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, preds_proba)

f1_scores = np.divide(
    2 * (precisions * recalls), (precisions + recalls),
    out=np.zeros_like(precisions), where=(precisions + recalls) != 0
)
best_threshold = thresholds[np.argmax(f1_scores)]

final_preds = (preds_proba >= best_threshold).astype(int)
print(f"최적 Threshold : {best_threshold:.4f}")

# ---- Test 평가 ----
test_preds_proba = model.predict_proba(X_test)[:, 1]
test_final_preds = (test_preds_proba >= best_threshold).astype(int)

print("\n[최종 테스트 결과]")
print(f"PR-AUC   : {average_precision_score(y_test, test_preds_proba):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, test_preds_proba):.4f}")
print(f"F1-Score : {f1_score(y_test, test_final_preds):.4f}")
print(f"Precision: {precision_score(y_test, test_final_preds):.4f}")
print(f"Recall   : {recall_score(y_test, test_final_preds):.4f}")

test_cm = confusion_matrix(y_test, test_final_preds)
print(f"TN: {test_cm[0][0]:,} | FP: {test_cm[0][1]:,}")
print(f"FN: {test_cm[1][0]:,} | TP: {test_cm[1][1]:,}")

최적 Threshold : 0.2233

[최종 테스트 결과]
PR-AUC   : 0.3248
ROC-AUC  : 0.7197
F1-Score : 0.3870
Precision: 0.2974
Recall   : 0.5541
TN: 47,505 | FP: 15,967
FN: 5,438 | TP: 6,757


In [17]:
# mean(y_pred)
final_preds.mean()

np.float64(0.3170509295805319)